# DiT Anomaly Detection â€” Quick Verification Notebook
**Group 6 | DATA-MSML 612 | University of Maryland**

This notebook **verifies pre-trained results**. No training required.

**Time:** ~25-35 minutes on Colab T4 (evaluation only, no training)

**Prerequisites:**
- `ALL_OUTPUT.zip` in your Google Drive root (contains 15 DiT + 1 UNet checkpoints, ~6.7 GB)
- Kaggle credentials in Colab Secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY`

**What this does:**
1. Restores pre-trained checkpoints from your Drive
2. Downloads MVTec AD dataset from Kaggle
3. Runs L2 evaluation on all 15 categories
4. Compares results against committed baseline numbers
5. Generates all report figures
6. Prints a PASS/FAIL verification table

**Expected results (from committed baseline):**
- Mean Image AUROC: 0.573
- Mean Pixel AUROC: 0.759

## Section 0 â€” Setup

In [ ]:
import subprocess, sys, torch

r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU   :', r.stdout.strip() or 'NO GPU DETECTED')
print('Python:', sys.version[:40])
print('Torch :', torch.__version__)
print('CUDA  :', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Connect a T4 GPU runtime before running this notebook.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, subprocess

REPO_URL = '<YOUR_REPO_URL>.git'
REPO_DIR = '/content/dit_anomaly'

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git','clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    print((r.stdout + r.stderr)[-400:])
else:
    r = subprocess.run(['git','pull','--rebase'], capture_output=True, text=True, cwd=REPO_DIR)
    print('Repo:', (r.stdout+r.stderr)[-200:].strip())

CODE_DIR = os.path.join(REPO_DIR, 'code')
os.chdir(CODE_DIR)
print('Working dir:', os.getcwd())

In [ ]:
import subprocess, sys, torch

_deps = ['einops','timm','pytorch-msssim','scikit-image','lpips']
r = subprocess.run(['pip','install','-q'] + _deps, capture_output=True, text=True)
print('Packages:', 'ok' if r.returncode == 0 else r.stderr[-200:])

sys.path.insert(0, '.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '../data/mvtec'
ALL_CATEGORIES = [
    'bottle','cable','capsule','carpet','grid',
    'hazelnut','leather','metal_nut','pill','screw',
    'tile','toothbrush','transistor','wood','zipper',
]
RESULTS_DIR = '..'

BASELINE = {
    'bottle':      {'image_auroc': 0.458, 'pixel_auroc': 0.846},
    'cable':       {'image_auroc': 0.502, 'pixel_auroc': 0.630},
    'capsule':     {'image_auroc': 0.572, 'pixel_auroc': 0.857},
    'carpet':      {'image_auroc': 0.562, 'pixel_auroc': 0.730},
    'grid':        {'image_auroc': 0.434, 'pixel_auroc': 0.683},
    'hazelnut':    {'image_auroc': 0.744, 'pixel_auroc': 0.836},
    'leather':     {'image_auroc': 0.658, 'pixel_auroc': 0.913},
    'metal_nut':   {'image_auroc': 0.593, 'pixel_auroc': 0.581},
    'pill':        {'image_auroc': 0.681, 'pixel_auroc': 0.776},
    'screw':       {'image_auroc': 0.560, 'pixel_auroc': 0.892},
    'tile':        {'image_auroc': 0.593, 'pixel_auroc': 0.775},
    'toothbrush':  {'image_auroc': 0.367, 'pixel_auroc': 0.763},
    'transistor':  {'image_auroc': 0.533, 'pixel_auroc': 0.580},
    'wood':        {'image_auroc': 0.745, 'pixel_auroc': 0.777},
    'zipper':      {'image_auroc': 0.592, 'pixel_auroc': 0.753},
}
TOLERANCE = 0.015   # results must be within this of baseline
print('Device:', DEVICE, '| Categories:', len(ALL_CATEGORIES))
print('Baseline loaded for', len(BASELINE), 'categories')

## Section 1” Credentials, Checkpoints & Dataset

In [ ]:
import os, json
from pathlib import Path

def _write_kaggle_json(username, key):
    p = Path('/root/.config/kaggle/kaggle.json')
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps({'username': username, 'key': key}))
    p.chmod(0o600)

_ok = False
try:
    from google.colab import userdata
    _u = userdata.get('KAGGLE_USERNAME')
    _k = userdata.get('KAGGLE_KEY')
    if _u and _k:
        _write_kaggle_json(_u, _k)
        print('Kaggle credentials set. Username:', _u)
        _ok = True
    else:
        raise ValueError('Secrets missing')
except Exception as _e:
    print('Colab Secrets unavailable ({}) -- trying .env'.format(_e))

if not _ok:
    _env = Path('.env')
    if _env.exists():
        for _line in _env.read_text().splitlines():
            if '=' in _line and not _line.startswith('#'):
                _kk, _vv = _line.split('=', 1)
                os.environ[_kk.strip()] = _vv.strip()
    _u = os.environ.get('KAGGLE_USERNAME','')
    _k = os.environ.get('KAGGLE_KEY','')
    if _u and _k:
        _write_kaggle_json(_u, _k)
        print('Credentials set from .env.')
        _ok = True

if not _ok:
    raise RuntimeError(
        'No Kaggle credentials found.\n'
        'Add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets (key icon in sidebar).'
    )

In [ ]:
import os, zipfile, shutil
from pathlib import Path

DRIVE_DIR      = '/content/drive/MyDrive'
LOCAL_CKPT_DIT = 'output/checkpoints'
os.makedirs(LOCAL_CKPT_DIT, exist_ok=True)

_restored = len(list(Path(LOCAL_CKPT_DIT).rglob('best.pt')))
print('Checkpoints already on disk:', _restored)

if _restored < 15:
    _drive_zips = sorted(Path(DRIVE_DIR).glob('*.zip'))
    print('Zips in Drive:', [f.name for f in _drive_zips])

    _candidates = ['dit_anomaly_checkpoints.zip','ALL_OUTPUT.zip',
                   'all_outputs.zip','all_output.zip']
    for _f in _drive_zips:
        if _f.name not in _candidates:
            _candidates.append(_f.name)

    for _zip_name in _candidates:
        _zip_path = Path(DRIVE_DIR) / _zip_name
        if not _zip_path.exists():
            continue
        with zipfile.ZipFile(_zip_path,'r') as _z:
            _pt_files = [n for n in _z.namelist() if n.endswith('best.pt')]
        if not _pt_files:
            continue
        print('Extracting {} checkpoints from {}...'.format(len(_pt_files), _zip_name))
        with zipfile.ZipFile(_zip_path,'r') as _z:
            for _pt in _pt_files:
                _parts = Path(_pt).parts
                try:
                    _idx = next(i for i,p in enumerate(_parts) if 'checkpoint' in p.lower())
                except StopIteration:
                    _idx = 0
                _rel = Path(*_parts[_idx:])
                _dest = Path(LOCAL_CKPT_DIT) / Path(*_rel.parts[1:])
                _dest.parent.mkdir(parents=True, exist_ok=True)
                with _z.open(_pt) as _src, open(_dest,'wb') as _dst:
                    shutil.copyfileobj(_src, _dst)
        _restored = len(list(Path(LOCAL_CKPT_DIT).rglob('best.pt')))
        if _restored >= 15:
            break

_cats_found = sorted([p.parent.name for p in Path(LOCAL_CKPT_DIT).rglob('best.pt')])
print('Checkpoints ready: {}/15'.format(len(_cats_found)))
print('Categories:', _cats_found)
assert len(_cats_found) >= 15, 'Expected 15 checkpoints. Check ALL_OUTPUT.zip is in Drive root.'

In [ ]:
import os, zipfile, shutil, subprocess
from pathlib import Path

DATASET_SLUG = 'ipythonx/mvtec-ad'
DL_DIR       = '/tmp/mvtec_dl'
EXTRACT_DIR  = '/tmp/mvtec_extract'
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(DL_DIR, exist_ok=True)

_missing = [c for c in ALL_CATEGORIES
            if not Path('{}/{}/train/good'.format(DATA_ROOT, c)).exists()]

if not _missing:
    print('All 15 categories already present. Skipping download.')
else:
    print('{} categories missing. Downloading {}...'.format(len(_missing), DATASET_SLUG))
    _r = subprocess.run(
        ['kaggle','datasets','download','-d', DATASET_SLUG,'-p', DL_DIR,'-q'],
        capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError('kaggle download failed:\n' + _r.stderr)

    _zips = list(Path(DL_DIR).glob('*.zip'))
    if not _zips:
        raise RuntimeError('No zip found. Contents: {}'.format(
            [p.name for p in Path(DL_DIR).iterdir()]))

    _zip_path = _zips[0]
    print('Downloaded {:.2f} GB. Extracting...'.format(_zip_path.stat().st_size/1e9))
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(_zip_path,'r') as _z:
        _z.extractall(EXTRACT_DIR)
    _zip_path.unlink()

    _hits = list(Path(EXTRACT_DIR).rglob('bottle/train/good'))
    if not _hits:
        raise RuntimeError('Cannot find bottle/train/good after extraction.')
    _data_src = _hits[0].parent.parent.parent
    for _cat in _missing:
        _src = _data_src / _cat
        _dst = Path(DATA_ROOT) / _cat
        if _src.exists():
            if _dst.exists(): shutil.rmtree(_dst)
            shutil.move(str(_src), str(_dst))
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)

_present = [c for c in ALL_CATEGORIES
            if Path('{}/{}/train/good'.format(DATA_ROOT, c)).exists()]
print('Dataset ready: {}/15 categories.'.format(len(_present)))
assert len(_present) == 15, 'Dataset incomplete: {}'.format(_present)

## Section 2” Evaluation (L2 scoring, all 15 categories)

In [ ]:
import json, os, torch, numpy as np
from pathlib import Path

from src.dataset import get_dataloaders
from src.diffusion import GaussianDiffusion, cosine_beta_schedule
from src.dit import DiT_Tiny
from src.scoring import FeatureExtractor
from src.evaluate import evaluate_category

_CKPT_DIR  = 'output/checkpoints'
_OUT_DIR   = 'output/verify_results'
os.makedirs(_OUT_DIR, exist_ok=True)

_betas     = cosine_beta_schedule(1000)
_diffusion = GaussianDiffusion(_betas, device=DEVICE)
_feat_ext  = FeatureExtractor().to(DEVICE)

verify_results = {}

print('{:<14} {:>10} {:>10}   {:>9} {:>9}   {:>6}'.format(
    'Category','Img AUROC','Pix AUROC','Base Img','Base Pix','Status'))
print('-' * 68)

for _cat in ALL_CATEGORIES:
    _cache = Path(_OUT_DIR) / '{}.json'.format(_cat)
    if _cache.exists():
        _r = json.load(open(_cache))
        if _r.get('image_auroc', 0) > 0:
            verify_results[_cat] = _r
            _b = BASELINE.get(_cat, {})
            _ok = (abs(_r['image_auroc'] - _b.get('image_auroc',0)) <= TOLERANCE and
                   abs(_r['pixel_auroc']  - _b.get('pixel_auroc',0))  <= TOLERANCE)
            print('{:<14} {:>10.4f} {:>10.4f}   {:>9.4f} {:>9.4f}   {:>6}'.format(
                _cat, _r['image_auroc'], _r['pixel_auroc'],
                _b.get('image_auroc',0), _b.get('pixel_auroc',0),
                'PASS' if _ok else 'WARN'))
            continue

    _ckpt = '{}/{}/best.pt'.format(_CKPT_DIR, _cat)
    if not os.path.exists(_ckpt):
        print('{:<14} MISSING CHECKPOINT'.format(_cat))
        continue

    print('{:<14} evaluating...'.format(_cat), end='', flush=True)
    _model = DiT_Tiny(img_size=128).to(DEVICE)
    _c = torch.load(_ckpt, map_location=DEVICE, weights_only=False)
    _model.load_state_dict(_c['model_state_dict'])

    _, _tl = get_dataloaders(DATA_ROOT, _cat, img_size=128, batch_size=8)
    _res = evaluate_category(_model, _diffusion, _tl, _feat_ext,
        device=DEVICE, t_partial=250, num_ddim_steps=50,
        alpha=0.5, img_size=128, scoring='l2')
    verify_results[_cat] = _res

    with open(_cache,'w') as _f:
        json.dump(_res, _f, indent=2)

    _b  = BASELINE.get(_cat, {})
    _ok = (abs(_res['image_auroc'] - _b.get('image_auroc',0)) <= TOLERANCE and
           abs(_res['pixel_auroc']  - _b.get('pixel_auroc',0))  <= TOLERANCE)
    print('\r{:<14} {:>10.4f} {:>10.4f}   {:>9.4f} {:>9.4f}   {:>6}'.format(
        _cat, _res['image_auroc'], _res['pixel_auroc'],
        _b.get('image_auroc',0), _b.get('pixel_auroc',0),
        'PASS' if _ok else 'WARN'))

    del _model
    torch.cuda.empty_cache()

print('-' * 68)
if verify_results:
    _im = float(np.mean([v['image_auroc'] for v in verify_results.values()]))
    _pm = float(np.mean([v['pixel_auroc']  for v in verify_results.values()]))
    _bim = float(np.mean([BASELINE[c]['image_auroc'] for c in verify_results if c in BASELINE]))
    _bpm = float(np.mean([BASELINE[c]['pixel_auroc']  for c in verify_results if c in BASELINE]))
    print('{:<14} {:>10.4f} {:>10.4f}   {:>9.4f} {:>9.4f}'.format(
        'MEAN', _im, _pm, _bim, _bpm))
    _passes = sum(
        1 for _cat in verify_results
        if _cat in BASELINE and
           abs(verify_results[_cat]['image_auroc'] - BASELINE[_cat]['image_auroc']) <= TOLERANCE and
           abs(verify_results[_cat]['pixel_auroc']  - BASELINE[_cat]['pixel_auroc'])  <= TOLERANCE
    )
    print()
    print('Verification: {}/{} categories PASS (tolerance +/-{})'.format(
        _passes, len(verify_results), TOLERANCE))

## Section 3” Figures & Visual Comparison

In [ ]:
import json, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

FIG_DIR = 'output/verify_figures'
Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

# Load committed results from repo for comparison
_rf = Path(RESULTS_DIR + '/results_l2')
l2_committed = json.load(open(_rf/'l2_summary.json'))['results']
pc_committed  = json.load(open(_rf/'patchcore_all.json'))['results']
ssim_all      = json.load(open(RESULTS_DIR+'/results/all_results_summary.json'))
pca  = {c: {'image_auroc': ssim_all[c]['PCA']['img']} for c in ssim_all if 'PCA' in ssim_all[c]}
ae   = {c: {'image_auroc': ssim_all[c]['Conv-AE']['img']} for c in ssim_all if 'Conv-AE' in ssim_all[c]}

_use = {c: verify_results.get(c, l2_committed.get(c, {})) for c in ALL_CATEGORIES}

METHODS = [
    ('DiT-Tiny (L2, verified)', _use,         '#2196F3'),
    ('PatchCore (committed)',   pc_committed,  '#4CAF50'),
    ('PCA (committed)',         pca,           '#FF9800'),
    ('Conv-AE (committed)',     ae,            '#9C27B0'),
]

x = np.arange(len(ALL_CATEGORIES))
nM, w = len(METHODS), 0.19

fig, axes = plt.subplots(2, 1, figsize=(19, 12))
for ai, (metric, ylabel) in enumerate([('image_auroc','Image-Level AUROC'),
                                        ('pixel_auroc','Pixel-Level AUROC')]):
    ax = axes[ai]
    for mi, (name, data, col) in enumerate(METHODS):
        vals = [data.get(c, {}).get(metric, np.nan) for c in ALL_CATEGORIES]
        offset = (mi - (nM-1)/2) * w
        ax.bar(x + offset, vals, w*0.9, label=name, color=col, alpha=0.85,
               edgecolor='white', linewidth=0.4)
        valid = [v for v in vals if not np.isnan(v)]
        if valid:
            ax.axhline(np.mean(valid), color=col, linestyle=':', lw=1.4, alpha=0.7)
    ax.axhline(0.5, color='#e53935', linestyle='--', lw=0.9, alpha=0.5, label='Random')
    ax.set_xticks(x)
    ax.set_xticklabels(ALL_CATEGORIES, rotation=38, ha='right', fontsize=8.5)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_ylim(0, 1.08)
    ax.set_title(ylabel + ' Verification â€” DiT-Tiny vs Baselines', fontsize=11)
    ax.legend(fontsize=9, ncol=2)
    ax.grid(axis='y', alpha=0.25)

fig.suptitle('MVTec AD Verification Results â€” Group 6, DATA-MSML 612', fontsize=12, fontweight='bold')
plt.tight_layout()
_out = '{}/auroc_comparison_verified.png'.format(FIG_DIR)
plt.savefig(_out, dpi=150, bbox_inches='tight')
plt.close()
from IPython.display import Image, display
display(Image(_out))
print('Saved:', _out)

## Section 4” Summary & Verification Report

In [ ]:
import json, numpy as np
from pathlib import Path

print('=' * 65)
print('FINAL VERIFICATION SUMMARY')
print('=' * 65)

# Load all committed results
_rf = Path(RESULTS_DIR + '/results_l2')
l2c  = json.load(open(_rf/'l2_summary.json'))['results']
pcc  = json.load(open(_rf/'patchcore_all.json'))['results']
tp   = json.load(open(_rf/'ablation_tpartial.json'))['sweep']
scr  = json.load(open(RESULTS_DIR+'/results/ablation_scoring.json'))['results']

print()
print('DiT-Tiny L2+max results (committed):')
print('  Mean Image AUROC: {:.3f}'.format(np.mean([v['image_auroc'] for v in l2c.values()])))
print('  Mean Pixel AUROC: {:.3f}'.format(np.mean([v['pixel_auroc']  for v in l2c.values()])))

print()
print('PatchCore WRN50-2 results (committed):')
print('  Mean Image AUROC: {:.3f}'.format(np.mean([v['image_auroc'] for v in pcc.values()])))
print('  Mean Pixel AUROC: {:.3f}'.format(np.mean([v['pixel_auroc']  for v in pcc.values()])))

print()
print('Scoring method ablation (hazelnut):')
for m, r in scr.items():
    print('  {:<8} Image={:.3f}  Pixel={:.3f}'.format(m.upper(), r['image_auroc'], r['pixel_auroc']))

print()
print('T_partial sweep (hazelnut, L2):')
print('  t=250 is optimal: img={:.3f}  pix={:.3f}'.format(
    tp['250']['image_auroc'], tp['250']['pixel_auroc']))

print()
print('Figures committed to figures/:')
for f in sorted(Path(RESULTS_DIR+'/figures').glob('*.png')):
    print('  ', f.name)

print()
if verify_results:
    _passes = sum(
        1 for _cat in verify_results
        if _cat in BASELINE and
           abs(verify_results[_cat]['image_auroc'] - BASELINE[_cat]['image_auroc']) <= TOLERANCE and
           abs(verify_results[_cat]['pixel_auroc']  - BASELINE[_cat]['pixel_auroc'])  <= TOLERANCE
    )
    print('VERIFICATION: {}/{} categories within tolerance +/-{}'.format(
        _passes, len(verify_results), TOLERANCE))
    if _passes == len(verify_results):
        print('ALL PASS - results are reproducible.')
    else:
        print('Some categories outside tolerance. Check GPU/random seed variation.')